In [18]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
!git clone https://github.com/mishitadudhaiya-code/NLP-Punctuation-Restoration-and-Translation

Cloning into 'NLP-Punctuation-Restoration-and-Translation'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.


In [21]:
%cd NLP-Punctuation-Restoration-and-Translation
!git fetch origin
!git switch --track origin/task-1-cadence-punctuation

/content/NLP-Punctuation-Restoration-and-Translation/NLP-Punctuation-Restoration-and-Translation
branch 'task-1-cadence-punctuation' set up to track 'origin/task-1-cadence-punctuation'.
Switched to a new branch 'task-1-cadence-punctuation'


In [22]:
!git branch

  main
* task-1-cadence-punctuation


In [24]:
PROJECT_DIR = "/content/drive/MyDrive/Research_Project/Model_1"

MODEL_DIR = os.path.join(PROJECT_DIR, "Cadence_Fast")
INPUT_DIR = os.path.join(PROJECT_DIR, "input_texts")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "punctuation_outputs")

for folder in [MODEL_DIR, INPUT_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project folders ready!")

Project folders ready!


In [25]:
import torch

assert torch.cuda.is_available(), (
    "GPU not detected. In Colab, select Runtime > Change runtime type > T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

GPU: Tesla T4
GPU memory: 14.56 GB


In [26]:
!pip show cadence-punctuation
!pip list | grep -i cadence

Name: cadence-punctuation
Version: 1.1.0
Summary: Multilingual punctuation restoration model for Indic languages
Home-page: https://github.com/AI4Bharat/Cadence
Author: AI4Bhārat
Author-email: opensource@ai4bharat.org
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: huggingface-hub, numpy, safetensors, torch, transformers
Required-by: 
cadence-punctuation                   1.1.0


In [31]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

Transformers: 4.57.1
PyTorch: 2.14.1+cu130


In [ ]:
!pip install -q --force-reinstall --no-deps transformers==4.57.1 tokenizers==0.22.1

In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=MODEL_DIR,
    local_files_only=True
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [38]:
import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 4.57.1
Hugging Face Hub: 1.33.0


In [41]:
import os
from pathlib import Path
from transformers import AutoTokenizer

# Find the tokenizer already downloaded in Google Drive
matches = list(Path(MODEL_DIR).rglob("tokenizer.json"))

print("Tokenizer files found:", len(matches))

if matches:
    TOKENIZER_DIR = str(matches[0].parent)
    print("Using local folder:", TOKENIZER_DIR)

    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_DIR,
        local_files_only=True
    )

    print("SUCCESS: Tokenizer loaded locally!")
else:
    print("Tokenizer file not found in MODEL_DIR.")

Tokenizer files found: 1
Using local folder: /content/drive/MyDrive/Research_Project/Model_1/Cadence_Fast/models--ai4bharat--Cadence-Fast/snapshots/8971c5011e4fba5dcfbcac52744587d7da605534
SUCCESS: Tokenizer loaded locally!


In [42]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(
    TOKENIZER_DIR,
    trust_remote_code=True,
    torch_dtype=torch.float32,
    local_files_only=True
)

model = model.to("cuda")
model.eval()

id2label = model.config.id2label

print("Cadence-Fast loaded successfully!")
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Cadence-Fast loaded successfully!
Device: cuda:0
Dtype: torch.float32


In [61]:

import torch
from transformers import AutoModel

model = AutoModel.from_pretrained(
    TOKENIZER_DIR,
    trust_remote_code=True,
    torch_dtype=torch.float32,
    local_files_only=True
)

model = model.to("cuda")
model.eval()

id2label = model.config.id2label

print("Cadence-Fast loaded successfully!")
print("Model:", type(model).__name__)
print("Device:", next(model.parameters()).device)


Cadence-Fast loaded successfully!
Model: Gemma3ForTokenClassification
Device: cuda:0


In [48]:
print("Model dtype:", next(model.parameters()).dtype)
print("Model device:", next(model.parameters()).device)

nan_params = sum(
    torch.isnan(p).sum().item()
    for p in model.parameters()
    if p.is_floating_point()
)

print("NaN values in model weights:", nan_params)

Model dtype: torch.float32
Model device: cuda:0
NaN values in model weights: 0


In [56]:
PROJECT_DIR = "/content/drive/MyDrive/Research_Project"

INPUT_DIR = os.path.join(PROJECT_DIR, "input_texts")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "punctuation_outputs")

print("Input folder exists:", os.path.exists(INPUT_DIR))
print("Output folder exists:", os.path.exists(OUTPUT_DIR))

files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(".txt")
])

print("\nAvailable input files:")
for i, f in enumerate(files):
    print(i, ":", f)

Input folder exists: True
Output folder exists: True

Available input files:
0 : sample-1.txt
1 : sample-2.txt
2 : sample-3.txt
3 : sample-4.txt
4 : sample-5.txt
5 : sample-6.txt
6 : sample-7.txt


In [57]:
FILE_INDEX = 0

filename = files[FILE_INDEX]
file_path = os.path.join(INPUT_DIR, filename)

with open(file_path, "r", encoding="utf-8") as f:
    original_text = f.read()

print("FILE:", filename)
print("Original characters:", len(original_text))

print("\n========== ORIGINAL TEXT ==========\n")
print(original_text[:2000])

print("\n========== PUNCTUATED TEXT ==========\n")
punctuated_text = punctuate_text(original_text)
print(punctuated_text)

FILE: sample-1.txt
Original characters: 3797

========== ORIGINAL TEXT ==========

भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज श्रीकृष्ण धर्मराज युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का नाम कांचीपुरम है यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं ऐसा कहा गया है कि चतुर्मास में इन सप्तपुरियों का दर्शन मोक्ष प्रदान करने वाला होता है हिंदू धर्म 

In [58]:

def punctuate_text_full(text, window_size=300, overlap=60):
    token_ids = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    total = len(token_ids)
    predictions = [None] * total

    stride = window_size - overlap
    bos_id = tokenizer.bos_token_id

    for start in range(0, total, stride):
        end = min(start + window_size, total)
        chunk = token_ids[start:end]

        # Add BOS token to each window
        window_ids = [bos_id] + chunk

        input_ids = torch.tensor(
            [window_ids], device="cuda"
        )
        attention_mask = torch.ones_like(input_ids)

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        chunk_predictions = (
            outputs.logits[0].argmax(dim=-1).tolist()
        )

        # Ignore overlap edges except at document boundaries
        valid_start = 0 if start == 0 else overlap // 2
        valid_end = (
            len(chunk) if end == total
            else len(chunk) - overlap // 2
        )

        for j in range(valid_start, valid_end):
            predictions[start + j] = chunk_predictions[j + 1]

        del outputs, input_ids, attention_mask

        if end == total:
            break

    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    result = []

    for token, pred in zip(tokens, predictions):
        if token == "\n":
            result.append("\n")
            continue

        result.append(token)

        punctuation = id2label[pred]
        if punctuation != "O":
            result.append(punctuation)

    return tokenizer.convert_tokens_to_string(result)


In [59]:

punctuated_text = punctuate_text_full(original_text)

output_path = os.path.join(
    OUTPUT_DIR, "sample-1_punctuated.txt"
)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(punctuated_text)

print("Original characters:", len(original_text))
print("Output characters:", len(punctuated_text))
print("Output saved:", output_path)
print("\nPUNCTUATED TEXT:\n")
print(punctuated_text)


Original characters: 3797
Output characters: 3858
Output saved: /content/drive/MyDrive/Research_Project/punctuation_outputs/sample-1_punctuated.txt

PUNCTUATED TEXT:

भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था। चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है। द्वारका चार धामों में एक धाम भी है। कहते हैं कि उज्जैन में महाकाल को नमस्कार कर लेने पर फिर मृत्यु की चिंता नहीं रहती। गढ़ मुक्तेश्वर के विषय में कहा जाता है कि महाभारत के विनाशकारी युद्ध के बाद योगीराज, श्रीकृष्ण, धर्मराज, युधिष्ठिर व अर्जुन के मन में युद्ध की विभीषिका और नरसंहार को देखकर भारी ग्लानि हुई। अयोध्या भगवान श्री राम के अवतार की पवित्र भूमि है। स्टेशन का नाम कांजीवरम जरूर है किंतु नगर का नाम कांचीपुरम है। यमुना तट पर भगवान श्रीकृष्ण की अवतार भूमि का यह पवित्र नगर मथुरा है। दक्षिण रेलवे के चेन्नै स्टेशन से धनुष्कोटि जाने वाली मुख्य लाइन पर चेन्नै से 35 मील की दूरी पर चेंगलपट स्टेशन है। ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं। 

In [60]:
import os

for filename in files:
    input_path = os.path.join(INPUT_DIR, filename)

    with open(input_path, "r", encoding="utf-8") as f:
        original = f.read()

    result = punctuate_text_full(original)

    output_name = filename.replace(".txt", "_punctuated.txt")
    output_path = os.path.join(OUTPUT_DIR, output_name)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(result)

    print(
        filename,
        "| Original:", len(original),
        "| Output:", len(result),
        "| Saved:", output_name
    )

print("\nAll seven files processed!")

sample-1.txt | Original: 3797 | Output: 3858 | Saved: sample-1_punctuated.txt
sample-2.txt | Original: 3825 | Output: 3887 | Saved: sample-2_punctuated.txt
sample-3.txt | Original: 3931 | Output: 3990 | Saved: sample-3_punctuated.txt
sample-4.txt | Original: 5428 | Output: 5532 | Saved: sample-4_punctuated.txt
sample-5.txt | Original: 4992 | Output: 5118 | Saved: sample-5_punctuated.txt
sample-6.txt | Original: 2330 | Output: 2375 | Saved: sample-6_punctuated.txt
sample-7.txt | Original: 3913 | Output: 3967 | Saved: sample-7_punctuated.txt

All seven files processed!
